# Dynamiqs type coverage utilities example

This notebook is a small and executable guide for writing typed Dynamiqs helper code. It distills the contributor guidance into a concrete example:

- accept broad user-facing inputs with `QArrayLike` and `ArrayLike`;
- normalize them immediately with `dq.asqarray()` and `jnp.asarray()`;
- return precise `QArray` and `jax.Array` values;
- use `typing.assert_type()` as a lightweight type-coverage assertion next to examples.

The example uses `dq.random`, qarray conversion utilities, and expectation-value utilities together.


## 1. Imports and public type aliases

Dynamiqs exports the qarray API from `dynamiqs.qarrays`, while JAX provides `Array` and `ArrayLike` for numerical values. `PRNGKeyArray` from jaxtyping documents random-key expectations.


In [ ]:
from __future__ import annotations

from collections.abc import Iterable
from dataclasses import dataclass
from typing import assert_type

import jax
import jax.numpy as jnp
from IPython.display import display as ipy_display
from jax import Array
from jax.typing import ArrayLike
from jaxtyping import PRNGKeyArray

import dynamiqs as dq
from dynamiqs.qarrays import QArray, QArrayLike

## 2. Typed wrappers around Dynamiqs conversion utilities

These wrappers mirror the style used in Dynamiqs public functions: keep the call signature permissive, then convert once at the boundary. This improves static coverage without making user code less ergonomic.


In [ ]:
def normalized_state(state: QArrayLike) -> QArray:
    """Convert a qarray-like state to a normalized Dynamiqs QArray."""
    qstate = dq.asqarray(state)
    return qstate.unit()


def centered_grid(points: ArrayLike) -> Array:
    """Convert array-like points to a centered JAX array."""
    values = jnp.asarray(points)
    return values - values.mean()


def random_density_matrix(key: PRNGKeyArray, dims: int | tuple[int, ...]) -> QArray:
    """Return a random normalized density matrix with a precise type."""
    return dq.random.dm(key, dims)

## 3. A typed observable utility

The helper below accepts any iterable of qarray-like observables. It converts each observable through `dq.asqarray()` and returns one stacked `jax.Array` of real expectation values. The `assert_type()` calls are executable no-ops, but static checkers can use them to verify the intended type contract.


In [ ]:
def real_expectations(observables: Iterable[QArrayLike], state: QArrayLike) -> Array:
    """Compute real expectation values for typed Dynamiqs inputs."""
    qstate = normalized_state(state)
    values = [
        dq.expect(dq.asqarray(observable), qstate).real for observable in observables
    ]
    return jnp.stack(values)


psi = normalized_state(dq.fock(4, 0) + dq.fock(4, 1))
observables = [dq.number(4), dq.position(4), dq.momentum(4)]
expectations = real_expectations(observables, psi)

assert_type(psi, QArray)
assert_type(expectations, Array)
ipy_display(expectations)

## 4. A small typed experiment object

Dataclasses are useful when notebook experiments grow into library code. The fields below document the experiment boundary, and the methods keep Dynamiqs-specific return types explicit.


In [ ]:
@dataclass(frozen=True)
class RandomStateExperiment:
    dims: int | tuple[int, ...]
    sample_points: ArrayLike

    def state(self, key: PRNGKeyArray) -> QArray:
        return dq.random.ket(key, self.dims)

    def density_matrix(self, key: PRNGKeyArray) -> QArray:
        return random_density_matrix(key, self.dims)

    def centered_points(self) -> Array:
        return centered_grid(self.sample_points)


key = jax.random.PRNGKey(7)
experiment = RandomStateExperiment(dims=4, sample_points=jnp.linspace(-2.0, 2.0, 9))
random_psi = experiment.state(key)
random_rho = experiment.density_matrix(key)
points = experiment.centered_points()

assert_type(random_psi, QArray)
assert_type(random_rho, QArray)
assert_type(points, Array)
ipy_display((random_psi.shape, random_rho.shape, points.shape))

## 5. Tips for using the above type-covered show-functions

Say if you want to move them into `dynamiqs/`,

1. keep the public function annotated,
2. use `QArrayLike`/`ArrayLike` for inputs that should accept Python, NumPy, JAX, or Dynamiqs values,
3. convert at the top of the function with `dq.asqarray()` or `jnp.asarray()`,
4. return `QArray` or `Array` rather than an untyped object,
5. run `task type`. See Below:



In [ ]:
!ty --version
!task type